"""

Paper 작성을 위해서, Locate Label 데이터를 EDA 하는 코드 

확인해볼 항목 

- 문장의 길이
- locate된 span 수
- span당 단어/토큰 수

일단은 token 단위로 진행 

"""

In [1]:
import pandas as pd
import numpy as np
from new_module.dev_utils.utils import *
from new_module.new_decode_utils import analyze_span_lengths_and_count

def read_gpt2_outputs(file_path):
    outputs = pd.read_json(file_path, lines=True)
    
    outputs = outputs.explode('generations',ignore_index=True)
    outputs['prompt']=outputs['prompt'].apply(lambda x: x['text'])
    
    keys=['text', 'tokens', 'locate_labels', 'tokens']
    
    for col in keys:
        outputs[col] = outputs['generations'].apply(lambda x: x.get(col,None))
        
    outputs.drop(columns=['generations'],inplace=True)
    return outputs
def read_jigsaw_outputs(file_path):
    outputs = pd.read_json(file_path, lines=True)
    
    keys=['text', 'toxicity', 'locate_labels', 'tokens']
    
    for col in keys:
        outputs[col] = outputs['source'].apply(lambda x: x.get(col,None))
        
    outputs.drop(columns=['source'],inplace=True)
    return outputs

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2-large')
tokenizer.add_special_tokens({"mask_token": '<mask>'})

def create_masked_text(tokens, token_labels):
    global tokenizer
    for i, label in enumerate(token_labels):
        if label == 1:
            tokens[i] = tokenizer.mask_token_id
    return tokenizer.decode(tokens, skip_special_tokens=False)

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /gpt2-large/resolve/main/tokenizer_config.json HTTP/11" 200 0


# Toxic spans - GPT

In [2]:
tox_gpt = read_gpt2_outputs('/data/hyeryung/mucoco/new_module/data/toxicity-avoidance/testset_gpt2_2500.jsonl')
tox_gpt = tox_gpt.dropna(subset=['locate_labels'])
tox_gpt['locate_labels_binary'] = tox_gpt['locate_labels'].apply(lambda x: [1 if i >= 0.5 else 0 for i in x])

In [3]:
# 샘플당 길이 
tox_gpt['tokens'].apply(len).describe()

count    115.000000
mean      23.130435
std        8.745447
min        1.000000
25%       17.000000
50%       24.000000
75%       29.500000
max       39.000000
Name: tokens, dtype: float64

In [7]:
# Span 당 token 수 분포 & 샘플당 span 수 분포
tox_gpt['masked_text'] = tox_gpt.apply(lambda x: create_masked_text(x['tokens'], x['locate_labels_binary']), axis=1)
tox_gpt['located_span_lengths'] = tox_gpt['masked_text'].apply(lambda x: analyze_span_lengths_and_count(x)[-1])
print('dist. of num of spans in a sample')
print(tox_gpt['located_span_lengths'].apply(len).describe())
print('-'*50)
print('mean span length', np.mean(sum(tox_gpt['located_span_lengths'].tolist(),[])))
print('min span length', np.min(sum(tox_gpt['located_span_lengths'].tolist(),[])))
print('max span length', np.max(sum(tox_gpt['located_span_lengths'].tolist(),[])))

dist. of num of spans in a sample
count    115.000000
mean       1.434783
std        1.085231
min        0.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        8.000000
Name: located_span_lengths, dtype: float64
--------------------------------------------------
mean span length 2.8848484848484848
min span length 1
max span length 10


In [ ]:
# tox_gpt_expanded = pd.read_excel('/data/hyeryung/mucoco/new_module/data/toxicity-avoidance/testset_gpt2_2500_locate_grad_metrics.xlsx')
# tox_gpt_expanded['labels'].apply(sum).value_counts()
# tox_gpt_expanded = tox_gpt_expanded[['prompt','text','tokens','words','char',
#                                      'tok2char','word2char','word2tok','tok2word','char2tok','char2word','labels','labels_binary','labels_binary_word', 'labels_binary_char',]]
# for col in ['tokens','words','char','tok2char','word2char','word2tok','tok2word','char2tok','char2word','labels','labels_binary','labels_binary_word', 'labels_binary_char',]:
#     tox_gpt_expanded[col] = tox_gpt_expanded[col].apply(eval)
# tox_gpt_expanded['text'].str.split('.').apply(len).describe()
# tox_gpt_expanded['words'].apply(len).describe()
# tox_gpt_expanded['tokens'].apply(len).describe()
# tox_gpt_expanded['labels_binary_word'].apply(sum).describe()

# Toxic spans - Jigsaw

In [8]:
tox_jigsaw = read_jigsaw_outputs('/data/hyeryung/mucoco/new_module/data/toxicity-avoidance/testset_jigsaw_1960.jsonl')
tox_jigsaw = tox_jigsaw.dropna(subset=['locate_labels']).copy()

In [9]:
tox_jigsaw['locate_labels_binary'] = tox_jigsaw['locate_labels'].apply(lambda x: [1 if i >= 0.5 else 0 for i in x])

In [10]:
# 샘플당 길이 
tox_jigsaw['tokens'].apply(len).describe()

count     40.000000
mean      60.950000
std       61.096372
min        5.000000
25%       20.500000
50%       34.500000
75%       71.000000
max      221.000000
Name: tokens, dtype: float64

In [ ]:
# Span 당 token 수 분포 & 샘플당 span 수 분포
tox_jigsaw['masked_text'] = tox_jigsaw.apply(lambda x: create_masked_text(x['tokens'], x['locate_labels_binary']), axis=1)
tox_jigsaw['located_span_lengths'] = tox_jigsaw['masked_text'].apply(lambda x: analyze_span_lengths_and_count(x)[-1])
print('dist. of num of spans in a sample')
print(tox_jigsaw['located_span_lengths'].apply(len).describe())
print('-'*50)
print('mean span length', np.mean(sum(tox_jigsaw['located_span_lengths'].tolist(),[])))
print('min span length', np.min(sum(tox_jigsaw['located_span_lengths'].tolist(),[])))
print('max span length', np.max(sum(tox_jigsaw['located_span_lengths'].tolist(),[])))

dist. of num of spans in a sample
count    40.000000
mean      2.200000
std       1.505545
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       7.000000
Name: located_span_lengths, dtype: float64
--------------------------------------------------
mean span length 2.727272727272727
min span length 1
max span length 14


# Inconsistent spans

In [12]:
nli = pd.read_json('/data/hyeryung/mucoco/new_module/data/NLI_locate/nli_contra_300_locate_labels.jsonl', lines=True)

In [17]:
# # 샘플당 identify된 token 수 
# nli.hypothesis_token_labels_binary.apply(sum).describe()

In [14]:
# 샘플당 길이 
nli.hypothesis_tokens.apply(len).describe()

count    300.000000
mean      11.973333
std        5.848073
min        3.000000
25%        8.000000
50%       11.000000
75%       14.000000
max       44.000000
Name: hypothesis_tokens, dtype: float64

In [16]:
# Span 당 token 수 분포 & 샘플당 span 수 분포
nli['masked_text'] = nli.apply(lambda x: create_masked_text(x['hypothesis_tokens'], x['hypothesis_token_labels_binary']), axis=1)
nli['located_span_lengths'] = nli['masked_text'].apply(lambda x: analyze_span_lengths_and_count(x)[-1])
print('dist. of num of spans in a sample')
print(nli['located_span_lengths'].apply(len).describe())
print('-'*50)
print('mean span length', np.mean(sum(nli['located_span_lengths'].tolist(),[])))
print('min span length', np.min(sum(nli['located_span_lengths'].tolist(),[])))
print('max span length', np.max(sum(nli['located_span_lengths'].tolist(),[])))


dist. of num of spans in a sample
count    300.000000
mean       1.253333
std        0.450740
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        3.000000
Name: located_span_lengths, dtype: float64
--------------------------------------------------
mean span length 3.0664893617021276
min span length 1
max span length 17
